# Lab 08 — Customer Segments — Mall K-Means
**Unsupervised Track** · Intermediate · ~50 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Apply K-Means clustering to customer data
2. Use elbow method and silhouette score
3. Standardise features before clustering
4. Interpret centroids as marketing personas

## Datasets (this folder)
- `Mall_Customers.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-08-kmeans-mall-segments/lab-08-kmeans-mall-segments.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Paste the lab token from the ViLabs page into `VL_TOKEN` in **Cell 0**, then run that cell first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`Mall_Customers.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-08-kmeans-mall-segments"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-08-kmeans-mall-segments/bundle/dataset.zip"
NEED = ["Mall_Customers.csv"]  # unzipped files used by the code below


import json as _json
import urllib.parse as _urlparse
import urllib.request as _urlrequest

VLABS_API = "https://prod-api.vilabs.app"
VL_TOKEN = ""  # paste the lab token shown on the ViLabs page after Start Lab

def vlabs_submit(answers):
    """Send this session's measured numbers. Call this from the solutions cell."""
    token = str(VL_TOKEN or "").strip()
    if not token:
        print("ViLabs: paste the lab token into VL_TOKEN in Cell 0, then re-run this cell.")
        return None
    base = VLABS_API.rstrip("/") + "/colab/labs/" + LAB_ID + "/session"
    challenge_url = base + "/challenge/?" + _urlparse.urlencode({"token": token})
    with _urlrequest.urlopen(challenge_url, timeout=30) as resp:
        challenge = _json.loads(resp.read().decode())
    missing = [name for name in challenge.get("variables") or [] if name not in answers]
    if missing:
        print("ViLabs: missing answers for", ", ".join(missing))
    payload = _json.dumps({"token": token, "answers": answers}).encode()
    req = _urlrequest.Request(
        base + "/validate/",
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        result = _json.loads(resp.read().decode())
    print("ViLabs seed", challenge.get("seed"), "validated", result.get("validated"), result.get("variables"))
    return result

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Customer Analytics: Mall Customer Segmentation with K-Means

> **Scenario:** A shopping mall's marketing team has collected data on 200 customers — age, annual income, and spending score — but no predefined segments. They want to discover natural groupings so they can tailor promotions: a "high-income big-spender" campaign makes no sense for a "budget-conscious retiree" segment. You will use K-Means clustering to find these segments and give each a marketing persona.
>
> **You will learn:** unsupervised learning vs supervised, why feature scaling matters, the elbow method for choosing k, silhouette score for cluster quality, and how to interpret cluster centroids as business personas.
> **Time:** ~50 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | Where it appears | Why it matters |
|---|---|---|
| Unsupervised learning | Section 1 | No labels — the algorithm finds structure on its own |
| StandardScaler | Section 1 | Income (15–137) would dominate age (18–70) without scaling |
| Elbow method | Section 2 | Pick k where adding more clusters stops helping |
| Silhouette score | Section 2 | Measures how well-separated clusters are (-1 to 1) |
| Centroid interpretation | Section 3 | Turns numbers into marketing personas |

---

## 1. Unsupervised Learning and Feature Scaling

In **supervised** learning (like the MNIST lab), every row comes with a label and the algorithm learns to predict it. In **unsupervised** learning there are no labels — the algorithm's job is to find hidden structure in the features themselves. **K-Means** is the most common unsupervised algorithm: it partitions points into `k` clusters by minimizing the squared distance from each point to its cluster center (centroid).

A critical preprocessing step: K-Means uses Euclidean distance, so features with larger scales dominate. Annual income ranges 15–137 while age ranges 18–70 — without scaling, income would pull clusters along its axis almost exclusively. **StandardScaler** transforms each feature to mean 0, standard deviation 1, putting all features on equal footing.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv("Mall_Customers.csv")
print(mall.head())
print(f"\nShape: {mall.shape}")
print(mall.describe().round(2))

# Select the three numeric features
features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
X = mall[features].to_numpy()

# Scale: zero mean, unit variance per feature
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nBefore scaling — feature means:", X.mean(axis=0).round(2))
print("Before scaling — feature stds: ", X.std(axis=0).round(2))
print("After scaling  — feature means:", X_scaled.mean(axis=0).round(2))
print("After scaling  — feature stds: ", X_scaled.std(axis=0).round(2))


**What to notice:**
1. The dataset has 200 rows and 5 columns; `CustomerID` and `Genre` are not used for clustering (ID is an identifier, Genre is categorical).
2. After scaling, every feature has mean ≈ 0 and std ≈ 1 — no single feature can dominate the distance calculation.
3. `Spending Score (1-100)` is already a 1–100 rating, but scaling still matters because its variance differs from the others.

> **Pitfall:** Fitting the scaler on the full dataset and then clustering leaks information. In a real pipeline you would fit on training data only — but with 200 rows and no prediction task, fitting on all data is acceptable here.

---

## 2. Choosing k: Elbow Method and Silhouette Score

How many clusters should we use? Two complementary diagnostics:

- **Elbow method:** Run K-Means for k = 1, 2, 3, … and plot the **inertia** (sum of squared distances to each centroid). Inertia always decreases as k grows, but the rate of decrease slows — the "elbow" where the curve bends is a good k.
- **Silhouette score:** For each point, measure how similar it is to its own cluster vs the nearest other cluster. Ranges from -1 (wrong cluster) to +1 (well-separated). Higher is better.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertias, silhouettes = [], []
K_RANGE = range(1, 11)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    if k > 1:
        silhouettes.append(silhouette_score(X_scaled, km.labels_))
    else:
        silhouettes.append(float("nan"))

print(" k   Inertia   Silhouette")
for k, ine, sil in zip(K_RANGE, inertias, silhouettes):
    print(f"{k:2d}  {ine:8.2f}  {sil:10.4f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(list(K_RANGE), inertias, "bo-")
plt.title("Elbow Method")
plt.xlabel("k"); plt.ylabel("Inertia")
plt.subplot(1, 2, 2)
plt.plot(list(K_RANGE)[1:], silhouettes[1:], "ro-")
plt.title("Silhouette Score")
plt.xlabel("k"); plt.ylabel("Silhouette")
plt.tight_layout()
plt.savefig("elbow_silhouette.png", dpi=100)
from IPython.display import display
display(plt.gcf())
print("Saved elbow_silhouette.png")


**What to notice:**
1. Inertia drops sharply from k=1 (600.0) to k=5 (168.3), then the drops get smaller — the elbow is around k=5.
2. Silhouette peaks at k=6 (≈ 0.428), slightly above k=5 (≈ 0.417) — both are reasonable choices.
3. Silhouette values around 0.4 indicate moderately separated structure — real-world customer data is rarely cleanly separable.

> **Pitfall:** Always set `n_init=10` (or higher) in sklearn's K-Means. The algorithm is sensitive to random centroid initialization; multiple runs and taking the best result makes your clusters reproducible.

---

## 3. Interpreting Centroids as Marketing Personas

Once we pick k, the real business value comes from **interpreting** the clusters. We transform the centroids back to the original scale (inverse transform) so we can read them as real ages, incomes, and spending scores. With k=5 we get the classic five mall segments:

In [ ]:
K = 5
km = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = km.fit_predict(X_scaled)

# Inverse-transform centroids to original units for interpretation
centroids = scaler.inverse_transform(km.cluster_centers_)
centroid_df = pd.DataFrame(centroids, columns=features)
centroid_df["Cluster"] = range(K)
print(centroid_df.round(1))

# Profile each cluster with real data
for c in range(K):
    sub = mall[labels == c]
    print(f"\nCluster {c} (n={len(sub)}):")
    print(f"  Avg age:    {sub['Age'].mean():.1f}")
    print(f"  Avg income: {sub['Annual Income (k$)'].mean():.1f}k")
    print(f"  Avg spend:  {sub['Spending Score (1-100)'].mean():.1f}")

# Visualize: income vs spending, colored by cluster
plt.figure(figsize=(7, 5))
for c in range(K):
    sub = mall[labels == c]
    plt.scatter(sub["Annual Income (k$)"], sub["Spending Score (1-100)"],
                label=f"Cluster {c}", s=30)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Mall Customer Segments (k=5)")
plt.legend()
plt.savefig("mall_segments.png", dpi=100)
from IPython.display import display
display(plt.gcf())
print("Saved mall_segments.png")


**What to notice:**
1. Cluster 2 (n=40): income ≈ 86k, spend ≈ 82 — **"Premium Customers"**: high earners who spend freely. Target with luxury goods and VIP offers.
2. Cluster 3 (n=39): income ≈ 86k, spend ≈ 19 — **"Frugal High-Earners"**: big paychecks but tight wallets. Target with investment products and value messaging.
3. Cluster 1 (n=54): age ≈ 25, spend ≈ 62 — **"Young Enthusiasts"**: early-career, moderate income, high spend. Target with trendy products and social media campaigns.
4. Cluster 0 (n=20): income ≈ 27k, spend ≈ 18 — **"Budget Shoppers"**: low income, low spend. Target with discounts and essentials.
5. Cluster 4 (n=47): age ≈ 56, spend ≈ 49 — **"Mainstream Seniors"**: older, middle income, moderate spend. Target with convenience and loyalty programs.

> **Pitfall:** Reading cluster profiles from the scaled centroids directly. A scaled income of 1.5 means nothing to a marketing team — always inverse-transform to original units before interpreting.

---

## Exercises (do these!)

### Exercise 1 — Elbow Method
Run K-Means for k = 1 through 10 and plot inertia vs k. Identify the elbow (the k where the curve bends most) and report the inertia at k=5.
*Expected: elbow at k=5; inertia at k=5 ≈ 168.25.*

**Follow-up:** What is the inertia at k=1? Check: 600.00 (equals 200 rows × 3 features, since each feature has variance 1 after scaling).

### Exercise 2 — Best k by Silhouette
Compute the silhouette score for k = 2 through 10. Which k has the highest score, and what is that score?
*Expected: k=6 with silhouette ≈ 0.4284.*

**Follow-up:** What is the silhouette score at k=5? Check: ≈ 0.4166.

### Exercise 3 — Name the Segments
Fit K-Means with k=5, inverse-transform the centroids, and write a one-line marketing persona for each cluster (e.g., "Young Enthusiasts: age ~25, high spend"). Report the cluster sizes.
*Expected: sizes 20, 54, 40, 39, 47 for clusters 0–4.*

**Follow-up:** Which cluster has the highest average income? Check: clusters 2 and 3 both ≈ 86.1k.

<details>
<summary>Hint</summary>

For Exercise 3, use `scaler.inverse_transform(km.cluster_centers_)` to get centroids in original units, then loop over clusters and print `mall[labels == c].mean()`.

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

mall = pd.read_csv("Mall_Customers.csv")
features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
X_scaled = StandardScaler().fit_transform(mall[features].to_numpy())

inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 11), inertias, "bo-")
plt.title("Elbow Method")
plt.xlabel("k"); plt.ylabel("Inertia")
plt.savefig("elbow.png", dpi=100)
from IPython.display import display
display(plt.gcf())
print(f"Inertia at k=5: {inertias[4]:.2f}")

# --- Solution 2 ---
from sklearn.metrics import silhouette_score

silhouettes = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    silhouettes[k] = silhouette_score(X_scaled, km.labels_)

best_k = max(silhouettes, key=silhouettes.get)
print(f"Best k: {best_k} with silhouette {silhouettes[best_k]:.4f}")
for k, s in silhouettes.items():
    print(f"  k={k}: {s:.4f}")

# --- Solution 3 ---
K = 5
scaler = StandardScaler().fit(mall[features].to_numpy())
X_scaled = scaler.transform(mall[features].to_numpy())
km = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = km.fit_predict(X_scaled)

centroids = scaler.inverse_transform(km.cluster_centers_)
for c in range(K):
    sub = mall[labels == c]
    print(f"Cluster {c} (n={len(sub)}): "
          f"age={sub['Age'].mean():.1f}, "
          f"income={sub['Annual Income (k$)'].mean():.1f}k, "
          f"spend={sub['Spending Score (1-100)'].mean():.1f}")

# --- Follow-up 1 ---
assert abs(inertias[0] - 600.0) < 0.01
print(f"PASS: inertia at k=1 = {inertias[0]:.2f}")

# --- Follow-up 2 ---
assert abs(silhouettes[5] - 0.4166) < 0.001
print(f"PASS: silhouette at k=5 = {silhouettes[5]:.4f}")

# --- Follow-up 3 ---
sizes = [int((labels == c).sum()) for c in range(K)]
assert sizes == [20, 54, 40, 39, 47]
print(f"PASS: cluster sizes = {sizes}")

vlabs_submit({
    "inertia_k1": inertias[0],
    "silhouette_k5": silhouettes[5],
})


### What to learn next
- Try DBSCAN, which finds clusters of arbitrary shape and doesn't require choosing k.
- Apply PCA before clustering to visualize high-dimensional segments in 2D.
- Use Gaussian Mixture Models for soft clustering (each customer gets a probability of belonging to each segment).
- Explore hierarchical clustering and dendrograms for a visual approach to choosing k.
- Move to Course 3 (Deep Learning) and see how neural networks learn features automatically instead of relying on hand-picked ones.

*Files in this folder: Mall_Customers.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
